# Nasdaq ITCH Feed Handler v1

In [ ]:
# imports

from pynq import Overlay, allocate, MMIO
import numpy as np
import time
import gzip

In [ ]:
# Constants

DMA_DATA    = 0x4040
GPIO0       = 0x4120
GPIO1       = 0x4121
GPIO2       = 0x4122    
PATH        = "/home/xilinx/jupyter_notebooks/Nasdaq/name.bit"
DATA_PATH   = "/home/xilinx/jupyter_notebooks/Nasdaq/12302019.NASDAQ_ITCH50.gz"

START_BYTES = 2


In [ ]:
# Overlay Load

ol = Overlay(PATH)
ol.download()
print("Overlay Loaded:")

print(ol.ip_dict.items())


writer = MMIO(ol.ip_dict(["data_handler"]["phys_addr"]),ol.ip_dict(["data_handler"]["addr_range"]))

In [ ]:
# Buffer Formation
buffer_0 = allocate(shape=10000, dtype=np.uint8)
buffer_1 = allocate(shape=10000, dtype=np.uint8)
buffer_choice = 0
buffer_count = 0


# Packet data

with gzip.open(DATA_PATH, "rb") as data:
    while True:
        start_msg = data.read(START_BYTES)

        if not start_msg:
            writer.write(DMA_DATA, buffer_1[:buffer_count]) if buffer_choice else writer.write(DMA_DATA, buffer_0[:buffer_count])
            print("All Data Parsed")
            break

        # Find length of instruction
        ins_len =   int.from_bytes(start_msg, byteorder='big')
        output_data = data.read(ins_len)
        
        # Buffer check (if full)
        if(buffer_count + ins_len >= 10000):
            writer.write(DMA_DATA, buffer_1[:buffer_count]) if buffer_choice else writer.write(DMA_DATA, buffer_0[:buffer_count])
            buffer_count = 0
            buffer_choice = 1- buffer_choice

        # Double buffer selection
        if(buffer_choice):
            tmp_data = np.frombuffer(output_data, dtype=np.uint8)
            buffer_1[buffer_count : buffer_count + ins_len] = tmp_data
            buffer_count += ins_len
        else:
            tmp_data = np.frombuffer(output_data, dtype=np.uint8)
            buffer_0[buffer_count : buffer_count + ins_len] = tmp_data
            buffer_count += ins_len

            
        


        
